In [7]:
from pageindex import PageIndexClient
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import os

In [8]:
load_dotenv()  # Load environment variables from .env file

True

In [9]:
pageindex_client = PageIndexClient(api_key=os.getenv("PAGEINDEX_API_KEY"))
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [10]:
data_dir = "./data"
data_file_name = "attention_is_all_you_need.pdf"
PDF_PATH = Path(data_dir) / data_file_name

In [ ]:
def check_document_status(doc_id: str):
    """
    Polls the PageIndex API to monitor document processing status.
    
    This function continuously checks the processing status of a document until
    it either completes successfully or fails. It implements a polling mechanism
    with a 10-second interval between status checks.
    
    Args:
        doc_id: Unique identifier for the document in PageIndex
        
    Returns:
        None - exits on failure, continues on success
    """
    print("Building tree index... ")
    import time

    while True:
        # Query PageIndex API for current document status
        status_result = pageindex_client.get_document(doc_id)
        status = status_result["status"]
        print(f"Current document status: {status}")

        # Check if processing completed successfully
        if status == "completed":
            print("Document processing completed.")
            break
        # Handle processing failure
        elif status == "failed":
            print("Document processing failed.")
            exit(1)
        # Continue polling if still processing
        else:
            print("Document is still processing. Waiting for 10 seconds before checking again...")
            time.sleep(10)

In [ ]:
def upload_document_to_pageindex(pdf_path: Path) -> str:
    """
    Uploads a PDF document to PageIndex and waits for processing completion.
    
    This function handles the complete upload workflow: submitting the document,
    receiving a document ID, and monitoring the processing status until completion.
    
    Args:
        pdf_path: Path object pointing to the PDF file to upload
        
    Returns:
        str: The document ID assigned by PageIndex after successful upload
    """
    # Submit PDF to PageIndex API
    print("Uploading the PDF document to PageIndex...")
    result = pageindex_client.submit_document(pdf_path)

    # Extract document ID from the response
    doc_id = result["doc_id"]
    print(f"Document uploaded with doc_id: {doc_id}")
    
    # Monitor processing status until completion
    check_document_status(doc_id)
    
    return doc_id

In [ ]:
import json

def get_document_tree(doc_id: str) -> list:
    """
    Retrieves or fetches the hierarchical document tree structure.
    
    This function implements a caching mechanism: it first checks if the document
    tree is already saved locally. If not found, it fetches from PageIndex API
    and caches the result for future use.
    
    Args:
        doc_id: Unique identifier for the document in PageIndex
        
    Returns:
        list: Hierarchical tree structure containing document sections and metadata
    """
    # Check if document tree exists in local cache
    if not Path.exists(Path(data_dir) / f"{doc_id}.json"):
        # Tree not cached locally - fetch from PageIndex API
        print(f"Document with doc_id {doc_id} does not exist locally. Fetching from PageIndex...")
        tree_result = pageindex_client.get_tree(doc_id, node_summary=True)
        pageindex_tree = tree_result.get("result",[])

        # Display tree metadata
        print(f"Top-level sections: {len(pageindex_tree)}")
        print("Raw tree structure (first node):")
        print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

        # Cache the tree structure locally for future use
        import json
        with open(Path(data_dir) / f"{doc_id}.json", "w") as f:
            json.dump(pageindex_tree, f)
    else:
        # Load tree from local cache
        with open(Path(data_dir) / f"{doc_id}.json", "r") as f:
            pageindex_tree = json.load(f)
            
    return pageindex_tree

In [ ]:
# Fetch all documents currently stored in PageIndex
response = pageindex_client.list_documents()
documents = response.get("documents", [])
doc_id = ""

# Search through existing documents to find our target PDF
for document in documents:
    # Check if document name matches our target file
    if document["name"] == data_file_name:
        doc_id = document["id"]
        break

# If document wasn't found in PageIndex, upload it now
if not doc_id:
    print(f"Document {data_file_name} not found in PageIndex. Uploading now...")
    doc_id = upload_document_to_pageindex(PDF_PATH)

In [ ]:
pageindex_tree = get_document_tree(doc_id)

In [ ]:
def print_tree(nodes, indent=0):
    """
    Recursively prints the hierarchical document tree structure.
    
    This function displays the document's table of contents with visual
    indentation to represent hierarchy. Each node shows its ID, title,
    and page number.
    
    Args:
        nodes: List of node dictionaries containing the tree structure
        indent: Current indentation level (increments with each recursive call)
        
    Returns:
        None - prints directly to console
    """
    for node in nodes:
        # Create visual indentation based on hierarchy level
        prefix = "  " * indent + ("--" if indent > 0 else "")
        
        # Extract page number (use "?" if not available)
        page = node.get("page_index", "?")
        
        # Print node information: [ID] Title (page number)
        print(f"{prefix}[{node['node_id']}] {node['title']} (p.{page})")
        
        # Recursively print child nodes with increased indentation
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

In [11]:
print_tree(pageindex_tree)

[0000] Attention Is All You Need (p.1)
  --[0001] Abstract (p.1)
  --[0002] 2 Background (p.2)
  --[0003] 3 Model Architecture (p.2)
    --[0004] 3.1 Encoder and Decoder Stacks (p.3)
    --[0005] 3.2 Attention (p.3)
      --[0006] 3.2.1 Scaled Dot-Product Attention (p.4)
      --[0007] 3.2.2 Multi-Head Attention (p.4)
      --[0008] 3.2.3 Applications of Attention in our Model (p.5)
    --[0009] 3.3 Position-wise Feed-Forward Networks (p.5)
    --[0010] 3.4 Embeddings and Softmax (p.5)
    --[0011] 3.5 Positional Encoding (p.6)
  --[0012] 4 Why Self-Attention (p.6)
  --[0013] 5 Training (p.7)
  --[0014] 6 Results (p.8)
    --[0015] 6.1 Machine Translation (p.8)
    --[0016] 6.2 Model Variations (p.8)
    --[0017] 6.3 English Constituency Parsing (p.9)
  --[0018] 7 Conclusion (p.10)
  --[0019] References (p.10)
  --[0020] Attention Visualizations (p.13)


In [ ]:
def get_relevant_nodes(query: str, tree: list, model: str = "gpt-4o") -> dict:
    """
    Identifies relevant document sections for a query using LLM reasoning.
    
    This function uses an LLM to analyze the document structure and intelligently
    select which sections are most likely to contain information relevant to
    answering the user's question.
    
    Args:
        query: User's question or search query
        tree: Hierarchical document structure from PageIndex
        model: OpenAI model to use for analysis
        
    Returns:
        dict: Contains "thinking" (reasoning process) and "node_list" (relevant node IDs)
    """

    # Build simplified tree representation for LLM analysis
    def build_simplified_structure(node_list, depth=0):
        """
        Recursively creates a lightweight version of the tree for LLM processing.
        
        Reduces the tree to only essential fields (id, title, page, summary)
        to minimize token usage while preserving structural information.
        """
        simplified = []
        for item in node_list:
            # Extract core node information
            node_info = {
                "node_id": item["node_id"],
                "title": item["title"],
                "page_index": item.get("page_index", "?")
            }

            # Add text preview if available for better context
            content = item.get("text", "")
            if content:
                node_info["summary"] = content[:200]

            # Recursively process child nodes to maintain hierarchy
            if "nodes" in item and item["nodes"]:
                node_info["children"] = build_simplified_structure(item["nodes"], depth + 1)

            simplified.append(node_info)

        return simplified

    # Create lightweight tree structure
    simplified_tree = build_simplified_structure(tree)

    # Construct analysis prompt for the LLM
    analysis_prompt = f"""You are given a query and a hierarchical knowledge graph in JSON format.

Your task:
- Read node summaries
- Identify the most relevant nodes for the question
- Return only node_ids as a list

Question:
{query}

Knowledge Graph:
{json.dumps(simplified_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<Your step-by-step reasoning here>",
  "node_list": ["node_id_1", "node_id_2", ...]  // List of node_ids that are relevant
}}
"""

    # Get LLM analysis of relevant sections
    completion = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": analysis_prompt}],
        response_format={"type": "json_object"}
    )

    # Parse and return the structured response
    analysis_result = json.loads(completion.choices[0].message.content)
    return analysis_result

In [ ]:
# Question about a specific technical detail from the Transformer paper
question = "How does the Transformer encode positional information, and what are the mathematical functions used?"
print(f"Running LLM Tree Search for query: '{question}'")

# Use LLM to analyze the document tree and identify relevant sections
result = get_relevant_nodes(question, pageindex_tree)

# Display the LLM's reasoning process
print("LLM Tree Search Result:")
print(result.get("thinking", "N/A"))
print()

# Show which document sections were identified as relevant
print(f"Relevant node IDs: {result.get('node_list', [])}")

In [ ]:
def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """
    Recursively searches the tree to find nodes matching target IDs.
    
    This function traverses the hierarchical tree structure and collects
    all nodes whose IDs match the provided target IDs. It explores both
    the current level and all nested child nodes.
    
    Args:
        tree: List of node dictionaries representing the document structure
        target_ids: List of node IDs to search for
        
    Returns:
        list: All nodes that match the target IDs
    """
    nodes = []
    
    # Iterate through each node in the current tree level
    for node in tree:
        # Check if current node matches any target ID
        if node["node_id"] in target_ids:
            nodes.append(node)
            
        # Recursively search child nodes if they exist
        if node.get("nodes"):
            nodes.extend(find_nodes_by_ids(node["nodes"], target_ids))
            
    return nodes

In [ ]:
def generate_answer_from_llm(question: str, nodes: list, model: str = "gpt-4o") -> str:
    """
    Generates a contextually grounded answer using retrieved document sections.
    
    This function takes the relevant document nodes identified by the tree search
    and uses an LLM to generate a comprehensive answer to the user's question
    based solely on the provided context.
    
    Args:
        question: The user's question
        nodes: List of relevant document nodes containing text content
        model: OpenAI model identifier for answer generation
        
    Returns:
        str: A comprehensive answer grounded in the provided context
    """
    # Handle edge case: no nodes retrieved
    if not nodes:
        return "No relevant sections found to answer the query."
    
    # Build context string from retrieved nodes
    section_contexts = []
    for node_data in nodes:
        # Create header with section title and page number
        section_header = f"[Section: '{node_data['title']}' | Page {node_data.get('page_index', '?')}]"
        
        # Extract section content
        section_content = node_data.get('text', 'Content not available')
        
        # Combine header and content
        section_contexts.append(f"{section_header}\n{section_content}")
    
    # Combine all sections with clear delimiters
    full_context = "\n\n--\n\n".join(section_contexts)
    
    # Craft instruction prompt for answer generation
    system_instruction = f"""Answer the question using ONLY the context below.
Do not use any external knowledge.

Question:
{question}

Context:
{full_context}

Answer:
"""
    
    # Generate answer via LLM
    llm_response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": system_instruction}]
    )
    
    # Extract and clean the generated answer
    generated_answer = llm_response.choices[0].message.content.strip()
    return generated_answer

In [ ]:
def vectorless_rag(query: str, tree: list) -> str:
    """
    End-to-end RAG pipeline without vector embeddings.
    
    This function implements a complete Retrieval-Augmented Generation workflow
    using PageIndex's hierarchical tree structure instead of traditional vector
    embeddings. It uses LLM reasoning to navigate the document structure.
    
    Pipeline steps:
    1. LLM analyzes the tree structure to identify relevant sections
    2. Retrieve the full content of identified nodes
    3. Generate a grounded answer using the retrieved content
    
    Args:
        query: User's question
        tree: Hierarchical document structure from PageIndex
        
    Returns:
        str: Generated answer based on retrieved document sections
    """

    # Display query header
    print("*" * 50)
    print(f"Query: {query}")
    print("*" * 50)

    # Step 1: Use LLM to identify relevant sections in the tree
    search_result = get_relevant_nodes(query, tree)
    node_ids = search_result.get("node_list", [])

    # Display LLM's reasoning and selected nodes
    print("LLM Tree Search Reasoning:")
    print(f"{search_result.get('thinking', '')[:200]}...")
    print(f"Relevant node IDs: {node_ids}")
    print("-" * 50)

    # Step 2: Retrieve full content of identified nodes
    relevant_nodes = find_nodes_by_ids(tree, node_ids)

    # Display retrieved sections
    print(f"Retrieved {len(relevant_nodes)} relevant nodes for answer generation.")
    for node in relevant_nodes:
        print(f"- {node['title']} (p.{node.get('page_index', '?')})")
        
    # Step 3: Generate answer using retrieved content
    answer = generate_answer_from_llm(query, relevant_nodes)

    # Display final answer
    print("-" * 50)
    print("Generated Answer:")
    print(answer)
    
    return answer

In [18]:
vectorless_rag(question, pageindex_tree)

**************************************************
Query: How does the Transformer encode positional information, and what are the mathematical functions used?
**************************************************
LLM Tree Search Reasoning:
To answer the question about how the Transformer encodes positional information and the mathematical functions used, I need to find sections in the knowledge graph that describe the positional encodin...
Relevant node IDs: ['0011']
--------------------------------------------------
Retrieved 1 relevant nodes for answer generation.
- 3.5 Positional Encoding (p.6)
--------------------------------------------------
Generated Answer:
The Transformer encodes positional information using sine and cosine functions of different frequencies. The mathematical functions used are:

For even indices \(2i\):
\[ PE_{(pos, 2i)} = \sin(pos / 10000^{2i / d_{\text{model}}}) \]

For odd indices \(2i + 1\):
\[ PE_{(pos, 2i + 1)} = \cos(pos / 10000^{2i / d_{\text{model}}}) 

'The Transformer encodes positional information using sine and cosine functions of different frequencies. The mathematical functions used are:\n\nFor even indices \\(2i\\):\n\\[ PE_{(pos, 2i)} = \\sin(pos / 10000^{2i / d_{\\text{model}}}) \\]\n\nFor odd indices \\(2i + 1\\):\n\\[ PE_{(pos, 2i + 1)} = \\cos(pos / 10000^{2i / d_{\\text{model}}}) \\]\n\nEach dimension of the positional encoding corresponds to a sinusoid with wavelengths forming a geometric progression from \\(2\\pi\\) to \\(10000 \\cdot 2\\pi\\).'

In [19]:
question = "What does the term “self-attention” mean?"

In [20]:
vectorless_rag(question, pageindex_tree)

**************************************************
Query: What does the term “self-attention” mean?
**************************************************
LLM Tree Search Reasoning:
To determine the most relevant nodes for the query 'What does the term “self-attention” mean?', I need to identify sections in the knowledge graph that specifically mention 'self-attention'. Starting ...
Relevant node IDs: ['0004', '0012']
--------------------------------------------------
Retrieved 2 relevant nodes for answer generation.
- 3.1 Encoder and Decoder Stacks (p.3)
- 4 Why Self-Attention (p.6)
--------------------------------------------------
Generated Answer:
The term “self-attention” refers to a mechanism that connects all positions in a sequence with a constant number of sequentially executed operations. It allows for learning long-range dependencies by shortening the path length between any combination of positions in the input and output sequences, making it easier to capture such dependencies

'The term “self-attention” refers to a mechanism that connects all positions in a sequence with a constant number of sequentially executed operations. It allows for learning long-range dependencies by shortening the path length between any combination of positions in the input and output sequences, making it easier to capture such dependencies compared to recurrent layers. This mechanism is faster than recurrent layers when the sequence length is smaller than the representation dimensionality, which is often the case in state-of-the-art models for tasks like machine translation.'

In [21]:
question = "What are the two main parts of the Transformer model?"

In [22]:
vectorless_rag(question, pageindex_tree)

**************************************************
Query: What are the two main parts of the Transformer model?
**************************************************
LLM Tree Search Reasoning:
To identify the two main parts of the Transformer model, we need to find sections in the knowledge graph related to the model's architecture. We should look for summaries mentioning core components or...
Relevant node IDs: ['0003', '0004']
--------------------------------------------------
Retrieved 2 relevant nodes for answer generation.
- 3 Model Architecture (p.2)
- 3.1 Encoder and Decoder Stacks (p.3)
--------------------------------------------------
Generated Answer:
The two main parts of the Transformer model are the encoder and the decoder.


'The two main parts of the Transformer model are the encoder and the decoder.'